# Analysis Orchestrator

> Notebook generated from [`examples/subgraphs/10_analysis_orchestrator.py`](../../examples/subgraphs/10_analysis_orchestrator.py).

| Item | Value |
|------|-------|
| **Dataset** | Business Intelligence Center (embedded — 6 mixed tasks) |
| **API key** | ✅ **No API keys required** — uses stubs/mocks. |

**Expected result:** 4 modes: simulation, flat vs hierarchical comparison, real LangGraph with stubs, and a hierarchy diagram.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
Analysis Orchestrator — Hierarchical Analytical Domain Orchestrator
====================================================================

Dataset:  Business Intelligence Center (custom)
          6 analytical tasks for a business intelligence center
          covering the 4 leaf agents of the domain: data_analyst,
          ml_pipeline, dev_pipeline and financial_analyst.

Pattern:  Hierarchical domain orchestrator (SPEC-042 Phase 40)
          ┌─────────────────────────────────────────────────────┐
          │  Full hierarchy (PRISMAL_HIERARCHICAL_MODE)         │
          │                                                     │
          │  root_supervisor                                    │
          │       │                                             │
          │  analysis_orchestrator ◄── entry point              │
          │       │                                             │
          │  analysis_supervisor  (domain LLM router)           │
          │  ┌────┼────────────────────┐                        │
          │  ▼    ▼         ▼          ▼                        │
          │  data_analyst  ml_pipeline  dev_pipeline             │
          │  financial_analyst                                  │
          │       │ (all return to the supervisor)              │
          │  analysis_supervisor → END                          │
          └─────────────────────────────────────────────────────┘

Difference vs a flat subgraph:
  - The orchestrator does NOT execute tasks: it only routes.
  - The domain supervisor uses the LLM to decide the leaf agent.
  - Once a leaf agent responds, the supervisor closes the turn (END).
  - The per-domain iteration counter prevents runaway loops (cap=8).

Modes:
  1. demo_simulation()      — simulates routing without LLM (deterministic)
  2. demo_comparison()      — compares routing vs manual routing
  3. demo_real_langgraph()  — real graph with stubs and MemorySaver
  4. demo_hierarchy()       — visualizes the full 3-level hierarchy

Usage:
  uv run python examples/subgraphs/10_analysis_orchestrator.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio
import textwrap
from dataclasses import dataclass, field
from typing import Any

## Dataset — Business Intelligence tasks

In [ ]:
@dataclass
class AnalyticsTask:
    id: str
    request: str
    domain: str  # data_analyst | ml_pipeline | dev_pipeline | financial_analyst
    complexity: str  # LOW | MEDIUM | HIGH
    expected_output: str
    context: str


TASKS: list[AnalyticsTask] = [
    AnalyticsTask(
        id="TASK-01",
        request="Analyze Q3 2025 sales by region and product. "
        "I need a ranking of the top 5 regions by YoY growth.",
        domain="data_analyst",
        complexity="MEDIUM",
        expected_output="DataFrame with sales per region, YoY growth and ranking",
        context="Table `sales` with columns: date, region, product, amount, units_sold",
    ),
    AnalyticsTask(
        id="TASK-02",
        request="Train a churn prediction model for premium users. "
        "Use the last 12 months of activity and export the pipeline to MLflow.",
        domain="ml_pipeline",
        complexity="HIGH",
        expected_output="XGBoost + LightGBM model; target AUC-ROC >= 0.85",
        context="Dataset: 45,000 premium users with 22 behavioral features",
    ),
    AnalyticsTask(
        id="TASK-03",
        request="Implement the REST endpoint /api/v2/reports in FastAPI. "
        "Must accept date and region filters, with pagination and Redis cache.",
        domain="dev_pipeline",
        complexity="HIGH",
        expected_output="FastAPI code + unit tests + OpenAPI documentation",
        context="Existing architecture: FastAPI 0.111, SQLAlchemy 2.0, Redis 7",
    ),
    AnalyticsTask(
        id="TASK-04",
        request="Analyze NVDA and TSLA: technical signals (RSI, MACD, Bollinger Bands) "
        "and fundamentals (P/E, EPS growth). Give me a BUY/HOLD/SELL recommendation.",
        domain="financial_analyst",
        complexity="HIGH",
        expected_output="Technical+fundamental report with recommendation and risk score",
        context="Yahoo Finance data; 6-month window; portfolio with current exposure 8%",
    ),
    AnalyticsTask(
        id="TASK-05",
        request="How many unique active users did we have in the last 30 days? "
        "Segment by plan (free/premium/enterprise) and compute ARPU.",
        domain="data_analyst",
        complexity="LOW",
        expected_output="DAU/MAU metrics by segment and monthly ARPU",
        context="Table `user_events` with user_id, event_type, timestamp, plan_type",
    ),
    AnalyticsTask(
        id="TASK-06",
        request="Refactor the authentication module (auth/jwt.py). "
        "Remove PyJWT 2.9 deprecation warnings and add PKCE support.",
        domain="dev_pipeline",
        complexity="MEDIUM",
        expected_output="Refactored code + regression tests + PR description",
        context="Current JWT: HS256; migrate to RS256 with 24h key rotation",
    ),
]

## Leaf agent stubs (simulation without LLM)

In [ ]:
BOLD = "\033[1m"
RESET = "\033[0m"
CYAN = "\033[96m"
GREEN = "\033[92m"
PURPLE = "\033[95m"
ORANGE = "\033[93m"
RED = "\033[91m"
DIM = "\033[2m"

AGENT_COLORS = {
    "data_analyst": CYAN,
    "ml_pipeline": PURPLE,
    "dev_pipeline": GREEN,
    "financial_analyst": ORANGE,
}

AGENT_ICONS = {
    "data_analyst": "📊",
    "ml_pipeline": "🤖",
    "dev_pipeline": "⚙️ ",
    "financial_analyst": "📈",
}


@dataclass
class AgentResult:
    agent: str
    task_id: str
    summary: str
    artifacts: list[str]
    metrics: dict[str, Any] = field(default_factory=dict)


def data_analyst_stub(task: AnalyticsTask) -> AgentResult:
    """Simulate data_analyst: SQL/DataFrame queries and statistical analysis."""
    if "sales" in task.request.lower() or "q3" in task.request.lower():
        return AgentResult(
            agent="data_analyst",
            task_id=task.id,
            summary="Query executed on `sales`. Top YoY regions: North (+34%), "
            "South (+28%), Center (+21%), East (+15%), West (+9%).",
            artifacts=["sales_q3_2025_regional.parquet", "ranking_yoy_growth.csv"],
            metrics={"rows_processed": 2_847_391, "query_time_ms": 312, "cache_hit": False},
        )
    return AgentResult(
        agent="data_analyst",
        task_id=task.id,
        summary="DAU/MAU computed: Free=12,450, Premium=3,280, Enterprise=890. "
        "ARPU: Free=$0.82, Premium=$45.20, Enterprise=$380.50.",
        artifacts=["user_metrics_30d.csv", "arpu_by_plan.json"],
        metrics={"rows_processed": 4_120_000, "query_time_ms": 489, "cache_hit": True},
    )


def ml_pipeline_stub(task: AnalyticsTask) -> AgentResult:
    """Simulate ml_pipeline: training, evaluation and export."""
    return AgentResult(
        agent="ml_pipeline",
        task_id=task.id,
        summary="Pipeline complete: XGBoost AUC-ROC=0.887, LightGBM AUC-ROC=0.891. "
        "Top features: days_since_login, sessions_7d, historical_upgrades. "
        "Model exported to MLflow run_id=a3f9c12e.",
        artifacts=[
            "churn_model_xgb.pkl",
            "churn_model_lgbm.pkl",
            "feature_importance.png",
            "shap_summary.html",
            "mlflow://run/a3f9c12e",
        ],
        metrics={
            "auc_roc": 0.891,
            "precision": 0.834,
            "recall": 0.792,
            "f1": 0.812,
            "train_samples": 36_000,
            "test_samples": 9_000,
        },
    )


def dev_pipeline_stub(task: AnalyticsTask) -> AgentResult:
    """Simulate dev_pipeline: PO → Architect → Developer → QA → Reviewer."""
    if "endpoint" in task.request.lower() or "fastapi" in task.request.lower():
        return AgentResult(
            agent="dev_pipeline",
            task_id=task.id,
            summary="PO defined user stories. Architect designed: FastAPI Router + "
            "ReportService + CacheLayer (Redis TTL=300s). Developer implemented "
            "415 lines. QA: 28 unit tests (100% pass), 4 integration tests. "
            "Reviewer: score=0.91 — APPROVED.",
            artifacts=[
                "api/v2/reports.py",
                "api/v2/schemas.py",
                "tests/test_reports_endpoint.py",
                "docs/openapi_reports.yaml",
            ],
            metrics={"review_score": 0.91, "test_coverage": 0.94, "loc": 415},
        )
    return AgentResult(
        agent="dev_pipeline",
        task_id=task.id,
        summary="Refactor complete: migrated HS256 → RS256, removed 7 "
        "PyJWT 2.9 DeprecationWarnings, PKCE flow implemented. "
        "22 regression tests green. Reviewer: score=0.88 — APPROVED.",
        artifacts=[
            "auth/jwt.py",
            "auth/pkce.py",
            "tests/test_auth_regression.py",
            "docs/auth_migration_rs256.md",
        ],
        metrics={"review_score": 0.88, "test_coverage": 0.97, "warnings_fixed": 7},
    )


def financial_analyst_stub(task: AnalyticsTask) -> AgentResult:
    """Simulate financial_analyst: technical + fundamental + risk analysis."""
    return AgentResult(
        agent="financial_analyst",
        task_id=task.id,
        summary="NVDA: RSI=68 (approaching overbought), MACD bullish cross, BB upper. "
        "P/E=38.2x, EPS growth +112% YoY → BUY (score 0.78). "
        "TSLA: RSI=44 (neutral), MACD bearish, BB mid. "
        "P/E=72.1x, EPS growth -8% YoY → HOLD (score 0.51). "
        "Combined portfolio risk: MEDIUM (VaR 95%=3.2%).",
        artifacts=[
            "NVDA_technical_report.pdf",
            "TSLA_technical_report.pdf",
            "portfolio_risk_assessment.json",
            "recommendation_summary.md",
        ],
        metrics={
            "NVDA_score": 0.78,
            "TSLA_score": 0.51,
            "portfolio_var_95": 0.032,
            "recommendation": {"NVDA": "BUY", "TSLA": "HOLD"},
        },
    )

## Hierarchical orchestrator simulator

In [ ]:
# Deterministic routing rules (replicates the LLM logic of the domain_supervisor)
_ROUTING_RULES: list[tuple[list[str], str]] = [
    # financial_analyst keywords
    (
        [
            "rsi",
            "macd",
            "bollinger",
            "p/e",
            "eps",
            "nvda",
            "tsla",
            "aapl",
            "stock_es",
            "exchange",
            "market",
            "buy",
            "sell",
            "hold",
            "technical",
            "fundamental",
            "forex",
            "crypto2",
            "crypto",
            "financial market",
            "stock",
        ],
        "financial_analyst",
    ),
    # ml_pipeline keywords
    (
        [
            "train",
            "model",
            "ml",
            "machine learning",
            "xgboost",
            "lightgbm",
            "random forest",
            "neural",
            "deep learning",
            "churn",
            "classification",
            "regression",
            "prediction",
            "automl",
            "mlflow",
            "sklearn",
            "auc",
            "f1-score",
        ],
        "ml_pipeline",
    ),
    # dev_pipeline keywords
    (
        [
            "implement",
            "develop",
            "endpoint",
            "api",
            "fastapi",
            "refactor_es",
            "code",
            "test",
            "pytest",
            "pr ",
            "pull request",
            "software architecture",
            "module",
            "class",
            "function",
            "refactor",
            "bug",
            "fix",
            "deploy",
            "ci/cd",
            "microservice",
            "rest service",
        ],
        "dev_pipeline",
    ),
    # data_analyst (default for analytical queries)
    (
        [
            "analyze",
            "query",
            "sql",
            "dataframe",
            "dashboard",
            "kpi",
            "metric",
            "active users",
            "sales",
            "ranking",
            "segment",
            "compute the",
            "how many",
            "report",
            "statistic",
            "distribution",
            "histogram",
            "correlation",
            "grouped",
            "sum",
            "average",
            "mean",
            "median",
        ],
        "data_analyst",
    ),
]


def simulate_domain_supervisor(request: str) -> str:
    """
    Simulate the routing decision of the analysis_supervisor.

    In production, this step calls the LLM with the system prompt of the
    domain_supervisor. Here we use keyword matching to
    reproduce the behavior without an API key.

    Returns:
        Leaf agent name: data_analyst | ml_pipeline |
        dev_pipeline | financial_analyst
    """
    lower = request.lower()
    scores: dict[str, int] = {
        "data_analyst": 0,
        "ml_pipeline": 0,
        "dev_pipeline": 0,
        "financial_analyst": 0,
    }
    for keywords, agent in _ROUTING_RULES:
        for kw in keywords:
            if kw in lower:
                scores[agent] += 1

    best = max(scores, key=lambda a: scores[a])
    return best if scores[best] > 0 else "data_analyst"


def run_leaf_agent(agent: str, task: AnalyticsTask) -> AgentResult:
    """Dispatch the task to the correct leaf agent."""
    stubs = {
        "data_analyst": data_analyst_stub,
        "ml_pipeline": ml_pipeline_stub,
        "dev_pipeline": dev_pipeline_stub,
        "financial_analyst": financial_analyst_stub,
    }
    return stubs[agent](task)


def print_task_header(task: AnalyticsTask) -> None:
    AGENT_COLORS[task.domain]
    print(f"\n{'░' * 64}")
    print(f"  {BOLD}{task.id}{RESET}  —  Complexity: {task.complexity}")
    print(f"  {DIM}{textwrap.shorten(task.request, 70)}{RESET}")
    print(f"{'░' * 64}")


def print_routing_step(from_node: str, to_node: str, reason: str = "") -> None:
    arrow = f"  {DIM}{'─' * 3}►{RESET}"
    print(f"\n  {BOLD}[analysis_supervisor]{RESET}")
    print(f"  routes: {BOLD}{from_node}{RESET} {arrow} {BOLD}{to_node}{RESET}")
    if reason:
        print(f"  reason: {DIM}{reason}{RESET}")


def print_agent_result(result: AgentResult) -> None:
    color = AGENT_COLORS[result.agent]
    icon = AGENT_ICONS[result.agent]
    print(f"\n  {icon} {color}{BOLD}[{result.agent}]{RESET}")
    # Wrap summary at 60 chars
    summary_lines = textwrap.wrap(result.summary, 58)
    for line in summary_lines:
        print(f"     {line}")
    if result.artifacts:
        print(
            f"  {DIM}  Artifacts: {', '.join(result.artifacts[:3])}"
            f"{'…' if len(result.artifacts) > 3 else ''}{RESET}"
        )
    if result.metrics:
        m_str = "  ".join(f"{k}={v}" for k, v in list(result.metrics.items())[:3])
        print(f"  {DIM}  Metrics:    {m_str}{RESET}")
    print(f"\n  {DIM}[analysis_supervisor]{RESET} {DIM}← returns from leaf agent → END{RESET}")

## DEMO 1 — Full orchestrator simulation

In [ ]:
def demo_simulation() -> None:
    """Run the full orchestration pipeline for the 6 tasks."""
    print(f"\n{'═' * 64}")
    print(f"  {BOLD}DEMO 1 — analysis_orchestrator simulation{RESET}")
    print("  6 analytical tasks → deterministic hierarchical routing")
    print(f"{'═' * 64}")

    routing_log: list[tuple[str, str, str]] = []

    for task in TASKS:
        print_task_header(task)

        # Step 1: analysis_supervisor decides the leaf agent
        routed_to = simulate_domain_supervisor(task.request)
        correct = routed_to == task.domain
        verdict = (
            f"{GREEN}✓ correct{RESET}" if correct else f"{RED}✗ expected: {task.domain}{RESET}"
        )

        print_routing_step(
            "analysis_orchestrator",
            routed_to,
            reason=f"Classified as domain '{routed_to}' by keywords",
        )
        print(f"  {DIM}Routing: {verdict}{RESET}")

        # Step 2: the leaf agent executes the task
        result = run_leaf_agent(routed_to, task)
        print_agent_result(result)
        routing_log.append((task.id, task.domain, routed_to))

    # Routing summary
    print(f"\n\n{'═' * 64}")
    print(f"  {BOLD}Routing summary{RESET}")
    print(f"{'═' * 64}")
    print(f"  {'ID':<10} {'Expected':<22} {'Routed':<22} {'OK'}")
    print(f"  {'─' * 9} {'─' * 21} {'─' * 21} {'─' * 4}")
    correct_count = 0
    for tid, expected, routed in routing_log:
        ok = expected == routed
        if ok:
            correct_count += 1
        mark = f"{GREEN}✓{RESET}" if ok else f"{RED}✗{RESET}"
        color_e = AGENT_COLORS[expected]
        color_r = AGENT_COLORS[routed]
        print(f"  {tid:<10} {color_e}{expected:<22}{RESET} {color_r}{routed:<22}{RESET} {mark}")

    accuracy = correct_count / len(routing_log) * 100
    print(
        f"\n  Routing accuracy: {BOLD}{accuracy:.0f}%{RESET} ({correct_count}/{len(routing_log)})"
    )
    print(f"{'═' * 64}")

## DEMO 2 — Hierarchical vs flat comparison

In [ ]:
def demo_comparison() -> None:
    """
    Show the difference between hierarchical mode and flat mode.

    Flat mode:        root_supervisor → leaf_agent (1 level, root knows ~12+ agents)
    Hierarchical mode: root_supervisor → analysis_orchestrator → leaf_agent
                     (2 levels, each supervisor knows ~3-4 agents)
    """
    print(f"\n{'═' * 64}")
    print(f"  {BOLD}DEMO 2 — Hierarchical vs Flat{RESET}")
    print(f"{'═' * 64}")

    print(f"""
  {BOLD}FLAT mode{RESET} (PRISMAL_HIERARCHICAL_MODE=false):

    root_supervisor knows 12+ agents:
    ┌────────────────────────────────────────────────┐
    │ researcher, rag_agent, cua_agent,               │
    │ coder, codeact_agent, planner, file_manager,    │
    │ skill_manager, data_analyst, dev_pipeline,      │
    │ ml_pipeline, financial_analyst, …               │
    └────────────────────────────────────────────────┘
    Problem: very long routing context → lower accuracy

  {BOLD}HIERARCHICAL mode{RESET} (PRISMAL_HIERARCHICAL_MODE=true):

    root_supervisor knows only 3 orchestrators:
    ┌─────────────────────────────────────────────┐
    │ research_orchestrator                        │
    │ engineering_orchestrator                     │
    │ analysis_orchestrator  ◄── this demo        │
    └─────────────────────────────────────────────┘
         │
         └─► analysis_supervisor knows only 4 leaves:
             ┌─────────────────────┐
             │ data_analyst        │ ← SQL/DataFrame/charts
             │ ml_pipeline         │ ← AutoML/training
             │ dev_pipeline        │ ← PO→Architect→Dev→QA
             │ financial_analyst   │ ← markets/technical/risk
             └─────────────────────┘

  Advantages of hierarchical mode:
    • Shorter routing prompts (~4 options vs ~12)
    • Higher routing accuracy (well-defined domains)
    • Scalability: adding leaves does not impact root
    • Per-domain iteration caps (8) avoid local loops
    • Per-domain isolated iteration counters
""")

    # Concrete routing example for TASK-04
    task = TASKS[3]  # financial_analyst
    print(f"  {BOLD}Concrete example — {task.id}:{RESET}")
    print(f"  Request: «{textwrap.shorten(task.request, 60)}»")
    print("""
  Flat mode:
    root_supervisor → financial_analyst  (1 hop, long prompt)

  Hierarchical mode:
    root_supervisor → analysis_orchestrator  [level-1 routing]
         analysis_supervisor → financial_analyst  [level-2 routing]
                  financial_analyst executes the task
         analysis_supervisor ← agent returns
    root_supervisor ← orchestrator returns

  Overhead: +1 LLM call for the intermediate routing
  Benefit: root_supervisor with minimal context → better accuracy
  Configuration: PRISMAL_HIERARCHICAL_MODE=true in .env
""")
    print(f"{'═' * 64}")

## DEMO 3 — Real LangGraph with stubs and MemorySaver

In [ ]:
async def demo_real_langgraph() -> None:
    """
    Build the analysis_orchestrator with stub nodes instead of
    the real pipelines, and run it with LangGraph + MemorySaver.

    Uses _make_definition() directly to inject stubs for the
    three pipeline subgraphs (dev_pipeline, ml_pipeline, financial_analyst).
    The data_analyst_node is also replaced to avoid LLM calls.
    """
    try:
        from typing import Annotated, TypedDict

        from langchain_core.messages import AIMessage, HumanMessage
        from langgraph.checkpoint.memory import MemorySaver

        from prismal.langgraph import END, StateGraph, add_messages

        from prismal.agents.domain_supervisor import make_domain_supervisor
        from prismal.agents.subgraphs.analysis_orchestrator.builder import (
            ANALYSIS_AGENTS,
            _analysis_router,
        )
        from prismal.agents.subgraphs.factory import SubgraphFactory
        from prismal.agents.subgraphs.registry import SubgraphDefinition
    except ImportError as e:
        print(f"\n  ⚠  Dependency not available: {e}")
        print("     Install with: uv pip install -e '.[dev,all]'")
        return

    print(f"\n{'═' * 64}")
    print(f"  {BOLD}DEMO 3 — Real LangGraph with stubs{RESET}")
    print("  Uses _make_definition() + MemorySaver, no real LLM")
    print(f"{'═' * 64}")

    # ── Define reduced state for the demo ─────────────────────────────────────
    class DemoState(TypedDict, total=False):
        messages: Annotated[list, add_messages]
        current_agent: str
        next_agent: str | None
        metadata: dict
        session_id: str

    # ── Stubs for the leaf agents ─────────────────────────────────────────────
    def _make_stub(agent_name: str, canned_response: str):
        """Create a stub node that returns a predefined response."""

        async def stub_node(state: DemoState) -> dict:
            print(f"  [{agent_name}] executing task…")
            return {
                "messages": [AIMessage(content=canned_response, name=agent_name)],
                "current_agent": agent_name,
            }

        stub_node.__name__ = agent_name
        return stub_node

    data_analyst_stub_node = _make_stub(
        "data_analyst",
        "Analysis complete: top 5 regions by YoY growth identified. "
        "North +34%, South +28%. DataFrame exported as sales_q3_2025_regional.parquet.",
    )
    ml_pipeline_stub_node = _make_stub(
        "ml_pipeline",
        "ML Pipeline complete: AUC-ROC=0.891 (LightGBM). "
        "Model exported to MLflow run_id=a3f9c12e.",
    )
    dev_pipeline_stub_node = _make_stub(
        "dev_pipeline",
        "Endpoint /api/v2/reports implemented. "
        "28 tests passing. Review score=0.91. PR #247 created.",
    )
    financial_analyst_stub_node = _make_stub(
        "financial_analyst",
        "NVDA: BUY (score=0.78). TSLA: HOLD (score=0.51). "
        "Portfolio VaR 95%=3.2%. Report generated.",
    )

    # ── Domain supervisor with deterministic routing (no LLM) ─────────────────
    # For the demo, we override the supervisor with one that uses keywords.
    async def demo_analysis_supervisor(state: DemoState) -> dict:
        """Demo version of analysis_supervisor: uses keywords, no LLM."""
        metadata = dict(state.get("metadata", {}))
        slot = dict(metadata.get("domain_analysis", {}))
        iteration = int(slot.get("iteration_count", 0))

        # Loop breaker: if the last message is from a leaf agent, END
        msgs = list(state.get("messages", []))
        if msgs:
            last = msgs[-1]
            last_agent = state.get("current_agent", "")
            if getattr(last, "type", "") == "ai" and last_agent != "analysis_supervisor":
                slot["iteration_count"] = iteration + 1
                metadata["domain_analysis"] = slot
                print(
                    f"  [analysis_supervisor] loop-breaker → END (agent {last_agent} already responded)"
                )
                return {
                    "current_agent": "analysis_supervisor",
                    "next_agent": None,
                    "metadata": metadata,
                }

        # Keyword routing
        human_msgs = [m for m in msgs if getattr(m, "type", "") == "human"]
        request = human_msgs[-1].content if human_msgs else ""
        routed = simulate_domain_supervisor(request)
        print(f"  [analysis_supervisor] routing → {AGENT_COLORS[routed]}{BOLD}{routed}{RESET}")

        slot["iteration_count"] = iteration + 1
        metadata["domain_analysis"] = slot
        return {"current_agent": "analysis_supervisor", "next_agent": routed, "metadata": metadata}

    # ── Build the graph ───────────────────────────────────────────────────────
    builder = StateGraph(DemoState)
    builder.add_node("analysis_supervisor", demo_analysis_supervisor)
    builder.add_node("data_analyst", data_analyst_stub_node)
    builder.add_node("ml_pipeline", ml_pipeline_stub_node)
    builder.add_node("dev_pipeline", dev_pipeline_stub_node)
    builder.add_node("financial_analyst", financial_analyst_stub_node)

    builder.set_entry_point("analysis_supervisor")
    builder.add_conditional_edges("analysis_supervisor", _analysis_router)
    for leaf in ANALYSIS_AGENTS:
        builder.add_edge(leaf, "analysis_supervisor")

    checkpointer = MemorySaver()
    graph = builder.compile(checkpointer=checkpointer)

    # ── Run 3 tasks ───────────────────────────────────────────────────────────
    demo_tasks = [TASKS[0], TASKS[1], TASKS[2]]  # data_analyst, ml, dev
    for i, task in enumerate(demo_tasks, 1):
        config = {"configurable": {"thread_id": f"analysis-demo-{task.id}"}}
        initial: DemoState = {
            "messages": [HumanMessage(content=task.request)],
            "current_agent": "",
            "next_agent": None,
            "metadata": {},
            "session_id": f"demo-{i}",
        }
        print(f"\n  {BOLD}Task {i}/{len(demo_tasks)}: {task.id}{RESET}")
        print(f"  {DIM}{textwrap.shorten(task.request, 60)}{RESET}")

        result = await graph.ainvoke(initial, config)

        final_msgs = result.get("messages", [])
        ai_msgs = [m for m in final_msgs if getattr(m, "type", "") == "ai"]
        if ai_msgs:
            print(f"  {GREEN}✓{RESET} Agent response:")
            print(f"  {DIM}{textwrap.shorten(ai_msgs[-1].content, 70)}{RESET}")
        meta = result.get("metadata", {})
        iters = meta.get("domain_analysis", {}).get("iteration_count", "?")
        print(f"  Domain supervisor iterations: {iters}")

    print(f"\n{'═' * 64}")
    print("  LangGraph demo completed.")
    print("  Key patterns demonstrated:")
    print("    • Loop-breaker: supervisor detects leaf response → END")
    print("    • Isolated counter: domain_analysis.iteration_count")
    print("    • Conditional router: _analysis_router(state) → str")
    print(f"{'═' * 64}")

## DEMO 4 — Diagram of the full 3-level hierarchy

In [ ]:
def demo_hierarchy() -> None:
    """Visualize the framework's full 3-level hierarchy."""
    print(f"\n{'═' * 64}")
    print(f"  {BOLD}DEMO 4 — Full 3-level hierarchy{RESET}")
    print("  PRISMAL_HIERARCHICAL_MODE=true")
    print(f"{'═' * 64}")

    print(f"""
  {BOLD}Level 0 — Root Supervisor{RESET}
  ┌──────────────────────────────────────────────────────┐
  │  root_supervisor                                      │
  │  Knows: 3 domain orchestrators                       │
  │  System prompt: ~300 tokens (vs ~900 in flat mode)   │
  └───────────┬──────────────┬──────────────┬────────────┘
              │              │              │
  {BOLD}Level 1 — Domain Orchestrators{RESET}
  ┌───────────┴───┐  ┌───────┴──────┐  ┌───┴────────────────┐
  │ research_     │  │ engineering_ │  │ {CYAN}analysis_{RESET}        │
  │ orchestrator  │  │ orchestrator │  │ {CYAN}orchestrator{RESET}     │
  │               │  │              │  │ ← this demo          │
  │ Cap: 8 iters  │  │ Cap: 8 iters │  │ {CYAN}Cap: 8 iters{RESET}    │
  └───────┬───────┘  └──────┬───────┘  └────────┬───────────┘
          │                 │                   │
  {BOLD}Level 2 — Leaf Agents{RESET}
  ┌───────┴───────┐  ┌──────┴───────┐  ┌────────┴──────────┐
  │ researcher    │  │ coder        │  │ {CYAN}data_analyst{RESET}    │
  │ rag_agent     │  │ codeact_agent│  │ {PURPLE}ml_pipeline{RESET}     │
  │ cua_agent     │  │ planner      │  │ {GREEN}dev_pipeline{RESET}    │
  │               │  │ file_manager │  │ {ORANGE}financial_analyst{RESET}│
  │               │  │ skill_manager│  │                    │
  └───────────────┘  └──────────────┘  └───────────────────┘

  {BOLD}analysis_orchestrator properties:{RESET}
    • Entry point:     analysis_supervisor (LangGraph node)
    • Conditional:     _analysis_router(state) → str
    • Leaf agents:     {ANALYSIS_AGENTS_STR}
    • Return edges:    each leaf → analysis_supervisor
    • Flat bypass:     PRISMAL_HIERARCHICAL_MODE=false
    • Checkpointer:    Per-orchestrator isolated SQLite
    • Registry:        SubgraphRegistry (idempotent)

  {BOLD}Public API:{RESET}
    from prismal.agents.subgraphs.analysis_orchestrator import (
        get_compiled_analysis_orchestrator,   # retorna CompiledStateGraph
        register_analysis_orchestrator,       # registra en SubgraphRegistry
    )

    # With stubs for testing:
    graph = await get_compiled_analysis_orchestrator(
        dev_pipeline_graph=my_dev_stub,
        ml_pipeline_graph=my_ml_stub,
        financial_analyst_graph=my_fin_stub,
    )

    # Production (lazy-builds the 3 pipelines):
    graph = await get_compiled_analysis_orchestrator()
""")

    # Show real routing for each task
    print(f"  {BOLD}Routing of the BI Center dataset:{RESET}")
    print(f"  {'ID':<10} {'Routed leaf agent':<22} {'Complexity'}")
    print(f"  {'─' * 9} {'─' * 21} {'─' * 10}")
    for task in TASKS:
        routed = simulate_domain_supervisor(task.request)
        color = AGENT_COLORS[routed]
        print(f"  {task.id:<10} {color}{routed:<22}{RESET} {task.complexity}")
    print(f"{'═' * 64}")


ANALYSIS_AGENTS_STR = "data_analyst, ml_pipeline, dev_pipeline, financial_analyst"

## main

In [ ]:
async def main() -> None:
    print(f"""
{"═" * 64}
  {BOLD}Analysis Orchestrator — Hierarchical Domain Orchestrator{RESET}
  Dataset: Business Intelligence Center (6 analytical tasks)
  Agents: data_analyst · ml_pipeline · dev_pipeline · financial_analyst
{"═" * 64}
  Available modes:
    1. Simulation       — full orchestration (no LLM)
    2. Comparison       — hierarchical vs flat + diagram
    3. Real LangGraph   — stubs + MemorySaver + compiled graph
    4. Hierarchy        — full 3-level map + API
    5. All              — runs 1 + 2 + 3 + 4
{"─" * 64}""")

    choice = input("  Select mode [1-5] (Enter = 1): ").strip() or "1"

    if choice == "1":
        demo_simulation()
    elif choice == "2":
        demo_comparison()
    elif choice == "3":
        await demo_real_langgraph()
    elif choice == "4":
        demo_hierarchy()
    elif choice == "5":
        demo_simulation()
        demo_comparison()
        await demo_real_langgraph()
        demo_hierarchy()
    else:
        print("  Invalid option — running default simulation.")
        demo_simulation()

    print(f"""
  {DIM}Production configuration:
    PRISMAL_HIERARCHICAL_MODE=true   → activates the 3 orchestrators
    PRISMAL_HIERARCHICAL_MODE=false  → flat mode (default)
    PRISMAL_HITL_ENABLED=true        → human approval in dev_pipeline{RESET}
""")


if __name__ == "__main__":
    asyncio.run(main())

---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()